# Rerank Matrix Missing Entry Analysis

This notebook measures how complete the stored rerank matrices are for each dataset/model. It surfaces missing document coverage (docs present in qrels but absent in the matrix) and missing pairwise entries (unordered pairs that only have one/both directions).

**Coverage definitions**
- Matrices are loaded from `data/external/reranking-matrices` (recursively).
- BM25 top-K: prefer cached runs in `data/external/beir/bm25-runs`; if missing, try to generate from local BEIR Lucene indexes (via pyserini) and cache them.
- Candidate doc set per query = docs in the matrix ∪ BM25 top-K docs.
- `ordered_coverage_bm25_union` = `ordered_entries / (|docs_union_bm25| * (|docs_union_bm25|-1))`; values < 1.0 mean missing pairwise comparisons for BM25 docs.
- `docs_only_in_bm25` highlights BM25 docs that never appear in the rerank matrix (where missing comparison keys come from).

In [13]:
from pathlib import Path
from collections import defaultdict
import pickle
import re
import sys
import json

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

sys.path.append('..')
from ireranker.config import EXTERNAL_DATA_DIR, PROJ_ROOT

BASE_DIR = EXTERNAL_DATA_DIR / 'reranking-matrices'
QRELS_BASE = EXTERNAL_DATA_DIR / 'beir'
BM25_RUN_DIR = QRELS_BASE / 'bm25-runs'

pd.set_option('display.max_rows', 20)
pd.set_option('display.precision', 3)

BASE_DIR

PosixPath('/home/jerefigo/Documents/UdeSA/Procesamiento_del_Lenguaje_Natural/IReranker/data/external/reranking-matrices')

In [14]:
def parse_matrix_path(path: Path) -> dict:
    """Extract dataset/model/timestamp from matrix filename."""
    stem = path.stem
    m = re.match(r"(?:(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_)?(.+)_([^_]+)$", stem)
    if m:
        timestamp, model, dataset = m.groups()
    else:
        timestamp = None
        bits = stem.split("_")
        dataset = bits[-1] if len(bits) > 1 else stem
        model = "_".join(bits[:-1]) or None
    return {
        "dataset": dataset,
        "model": model,
        "timestamp": timestamp,
        "rel_path": path.relative_to(PROJ_ROOT),
    }


def _load_queries(dataset: str) -> dict[str, str]:
    """Read queries.jsonl into {qid: text}."""
    q_path = QRELS_BASE / dataset / "queries.jsonl"
    if not q_path.exists():
        return {}
    queries: dict[str, str] = {}
    with q_path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue
            qid = str(obj.get("_id") or obj.get("id") or "").strip()
            text = obj.get("text") or obj.get("query")
            if qid and text:
                queries[qid] = text
    return queries


def _generate_bm25_run(dataset: str, top_k: int = 100, restrict_qids: set[str] | None = None) -> dict[str, list[str]]:
    """Generate BM25 run from local Lucene index if pyserini is available."""
    try:
        from pyserini.search.lucene import LuceneSearcher  # type: ignore
    except Exception as e:
        print(f"[BM25] pyserini unavailable for {dataset}: {e}")
        return {}

    index_dir = QRELS_BASE / dataset / "lucene-index"
    if not index_dir.exists():
        print(f"[BM25] Missing Lucene index for {dataset} at {index_dir}")
        return {}

    queries = _load_queries(dataset)
    if restrict_qids:
        queries = {k: v for k, v in queries.items() if k in restrict_qids}
    if not queries:
        print(f"[BM25] No queries found for {dataset}")
        return {}

    searcher = LuceneSearcher(str(index_dir))
    runs: dict[str, list[str]] = {}
    for qid, text in tqdm(queries.items(), desc=f"BM25 search {dataset}", leave=False):
        hits = searcher.search(text, k=top_k)
        runs[qid] = [h.docid for h in hits][:top_k]
    return runs


def load_bm25_run(dataset: str, top_k: int = 100, restrict_qids: set[str] | None = None) -> dict[str, list[str]]:
    """Load cached BM25 run; if missing, try to generate from Lucene index and cache it."""
    BM25_RUN_DIR.mkdir(parents=True, exist_ok=True)
    run_path = BM25_RUN_DIR / f"run.beir.bm25-flat.{dataset}.txt"

    runs: dict[str, list[str]] = {}
    if run_path.exists():
        with run_path.open("r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6:
                    continue
                qid, _, doc_id, rank, score, _ = parts[:6]
                if restrict_qids and qid not in restrict_qids:
                    continue
                lst = runs.setdefault(qid, [])
                if len(lst) < top_k:
                    lst.append(doc_id)
        if runs:
            return runs

    generated = _generate_bm25_run(dataset, top_k=top_k, restrict_qids=restrict_qids)
    if generated:
        try:
            with run_path.open("w", encoding="utf-8") as f:
                for qid, doc_ids in generated.items():
                    for rank, doc_id in enumerate(doc_ids[:top_k], start=1):
                        f.write(f"{qid} Q0 {doc_id} {rank} {top_k - rank + 1:.4f} bm25
")
            print(f"[BM25] Cached run to {run_path}")
        except Exception as e:
            print(f"[BM25] Failed to cache run for {dataset}: {e}")
    return generated


def analyze_matrix_file(path: Path, bm25_lookup: dict[str, list[str]], bm25_top_k: int = 100):
    """Compute per-query coverage metrics for a rerank matrix vs BM25 top-K."""
    with path.open("rb") as f:
        matrix = pickle.load(f)

    per_q: dict[str, dict] = {}
    for key in matrix.keys():
        if not isinstance(key, tuple) or len(key) != 3:
            continue
        qid, doc_a, doc_b = key
        if not isinstance(qid, str) or not isinstance(doc_a, str) or not isinstance(doc_b, str):
            continue
        qstat = per_q.setdefault(qid, {"docs": set(), "ordered": 0, "dirs": defaultdict(set)})
        if doc_a == doc_b:
            continue
        qstat["ordered"] += 1
        qstat["docs"].update([doc_a, doc_b])
        unordered = tuple(sorted((doc_a, doc_b)))
        orient = "ab" if (doc_a, doc_b) == unordered else "ba"
        qstat["dirs"][unordered].add(orient)

    rows = []
    for qid, qstat in per_q.items():
        matrix_docs = qstat["docs"]
        dir_map = qstat["dirs"]
        unordered_pairs = len(dir_map)
        bidirectional = sum(1 for dirs in dir_map.values() if len(dirs) == 2)
        single_direction = sum(1 for dirs in dir_map.values() if len(dirs) == 1)
        doc_count_matrix = len(matrix_docs)
        expected_ordered_matrix = doc_count_matrix * (doc_count_matrix - 1)
        ordered_coverage_matrix = (
            qstat["ordered"] / expected_ordered_matrix if expected_ordered_matrix > 0 else 1.0
        )

        bm25_docs = set(bm25_lookup.get(qid, [])[:bm25_top_k]) if bm25_lookup else set()
        docs_union_bm25 = matrix_docs | bm25_docs
        union_count = len(docs_union_bm25)
        expected_ordered_union = union_count * (union_count - 1)
        ordered_coverage_union = (
            qstat["ordered"] / expected_ordered_union if expected_ordered_union > 0 else 1.0
        )

        rows.append(
            {
                "query_id": qid,
                "docs_in_matrix": doc_count_matrix,
                "docs_in_bm25_topk": len(bm25_docs),
                "docs_union_bm25": union_count,
                "docs_only_in_bm25": len(bm25_docs - matrix_docs),
                "docs_only_in_matrix": len(matrix_docs - bm25_docs),
                "ordered_entries": qstat["ordered"],
                "expected_ordered_matrix": expected_ordered_matrix,
                "expected_ordered_union": expected_ordered_union,
                "missing_ordered_matrix": max(expected_ordered_matrix - qstat["ordered"], 0),
                "missing_ordered_union": max(expected_ordered_union - qstat["ordered"], 0),
                "ordered_coverage_matrix": ordered_coverage_matrix,
                "ordered_coverage_bm25_union": ordered_coverage_union,
                "unordered_pairs_with_entries": unordered_pairs,
                "bidirectional_pairs": bidirectional,
                "single_direction_pairs": single_direction,
                "missing_unordered_pairs": max(
                    doc_count_matrix * (doc_count_matrix - 1) // 2 - unordered_pairs, 0
                ),
            }
        )

    per_query_df = pd.DataFrame(rows)
    summary = {
        "path": path,
        "entries": len(matrix),
        "queries": len(per_query_df),
    }
    if not per_query_df.empty:
        summary.update(
            {
                "avg_docs_in_matrix": per_query_df["docs_in_matrix"].mean(),
                "avg_docs_in_bm25_topk": per_query_df["docs_in_bm25_topk"].mean(),
                "avg_docs_only_in_bm25": per_query_df["docs_only_in_bm25"].mean(),
                "avg_docs_only_in_matrix": per_query_df["docs_only_in_matrix"].mean(),
                "pct_queries_with_bm25_only_docs": float(
                    (per_query_df["docs_only_in_bm25"] > 0).mean() * 100
                ),
                "pct_queries_missing_reverse": float(
                    (per_query_df["single_direction_pairs"] > 0).mean() * 100
                ),
                "mean_ordered_cov_matrix": per_query_df["ordered_coverage_matrix"].mean(),
                "mean_ordered_cov_bm25_union": per_query_df["ordered_coverage_bm25_union"].mean(),
                "mean_missing_unordered_pairs": per_query_df["missing_unordered_pairs"].mean(),
                "mean_missing_ordered_union": per_query_df["missing_ordered_union"].mean(),
            }
        )
    return per_query_df, summary

## Select matrices to analyze
Set filters if you only want to process a subset; leave empty lists to scan everything.

In [15]:
DATASET_FILTER = ["webis-touche2020"]  # e.g., ["trec-covid", "nfcorpus"]
MODEL_FILTER = []    # optional substring match on model name, e.g., ["xl", "llama"]

matrix_paths = sorted(BASE_DIR.rglob("*.pkl"))
selected = []
for path in matrix_paths:
    meta = parse_matrix_path(path)
    if DATASET_FILTER and meta["dataset"] not in DATASET_FILTER:
        continue
    if MODEL_FILTER and not any(token in (meta["model"] or "") for token in MODEL_FILTER):
        continue
    selected.append((path, meta))

print(f"Found {len(selected)} matrices after filtering.")
if selected:
    print("Example:", selected[0][0])
BM25_TOP_K = 100  # how many BM25 hits to check per query

Found 2 matrices after filtering.
Example: /home/jerefigo/Documents/UdeSA/Procesamiento_del_Lenguaje_Natural/IReranker/data/external/reranking-matrices/Reranking/large/2024_09_23_19_33_41_flan-t5-large_webis-touche2020.pkl


## Run coverage analysis

In [16]:
records = []
details = {}

for path, meta in tqdm(selected, desc='Analyzing matrices'):
    bm25_lookup = load_bm25_run(meta["dataset"], top_k=BM25_TOP_K)
    per_query_df, summary = analyze_matrix_file(path, bm25_lookup, bm25_top_k=BM25_TOP_K)
    summary.update(meta)
    summary["bm25_available"] = bool(bm25_lookup)
    summary["entries_per_query_mean"] = (
        per_query_df["ordered_entries"].mean() if not per_query_df.empty else 0
    )
    summary["file"] = str(summary.pop("path").relative_to(PROJ_ROOT))
    records.append(summary)
    details[summary["file"]] = {"meta": meta, "per_query": per_query_df}

summary_df = pd.DataFrame(records)
if not summary_df.empty:
    summary_df = summary_df.sort_values(by=["dataset", "model", "file"]).reset_index(drop=True)

summary_df.head(20)

Analyzing matrices: 100%|██████████| 2/2 [00:05<00:00,  2.85s/it]


,entries,queries,avg_docs_in_matrix,avg_docs_union,avg_docs_only_in_qrels,avg_docs_only_in_matrix,pct_queries_with_qrels_only_docs,pct_queries_missing_reverse,mean_ordered_cov_matrix,mean_ordered_cov_union,mean_missing_unordered_pairs,mean_missing_ordered_union,dataset,model,timestamp,rel_path,qrels_available,entries_per_query_mean,file
0,485100,49,100.0,133.061,33.061,87.878,100.0,0.0,1.0,0.565,0.0,7686.857,webis-touche2020,flan-t5-large,2024_09_23_19_33_41,data/external/reranking-matrices/Reranking/lar...,True,9900.0,data/external/reranking-matrices/Reranking/lar...
1,485100,49,100.0,133.061,33.061,87.878,100.0,0.0,1.0,0.565,0.0,7686.857,webis-touche2020,flan-t5-xl,2024_10_12_13_27_22,data/external/reranking-matrices/Reranking/xl/...,True,9900.0,data/external/reranking-matrices/Reranking/xl/...


## Files with the largest gaps
Lower `mean_ordered_cov_union` and higher `avg_docs_only_in_qrels` point to datasets where many qrels docs never appear in the matrix (which causes missing comparison keys).

In [17]:
if summary_df.empty:
    print("No matrices analyzed yet.")
else:
    cols = [
        "dataset",
        "model",
        "file",
        "entries",
        "queries",
        "avg_docs_in_matrix",
        "avg_docs_in_bm25_topk",
        "avg_docs_only_in_bm25",
        "pct_queries_with_bm25_only_docs",
        "pct_queries_missing_reverse",
        "mean_ordered_cov_bm25_union",
    ]
    display(summary_df[cols].sort_values("avg_docs_only_in_bm25", ascending=False).head(10))

,dataset,model,file,entries,queries,avg_docs_in_matrix,avg_docs_union,avg_docs_only_in_qrels,pct_queries_with_qrels_only_docs,pct_queries_missing_reverse,mean_ordered_cov_union
0,webis-touche2020,flan-t5-large,data/external/reranking-matrices/Reranking/lar...,485100,49,100.0,133.061,33.061,100.0,0.0,0.565
1,webis-touche2020,flan-t5-xl,data/external/reranking-matrices/Reranking/xl/...,485100,49,100.0,133.061,33.061,100.0,0.0,0.565


## Per-query detail for the worst file
Shows which queries drive the coverage issues (sorted by missing ordered pairs when using the matrix ∪ qrels candidate set).

In [18]:
if summary_df.empty:
    print("No matrices analyzed yet.")
else:
    target_file = summary_df.sort_values("avg_docs_only_in_bm25", ascending=False).iloc[0]["file"]
    print(f"Inspecting: {target_file}")
    per_query = details[target_file]["per_query"].copy()
    display(
        per_query
        .sort_values("docs_only_in_bm25", ascending=False)
        [[
            "query_id",
            "docs_in_matrix",
            "docs_in_bm25_topk",
            "docs_only_in_bm25",
            "ordered_entries",
            "missing_ordered_union",
            "single_direction_pairs",
        ]]
        .head(15)
    )

Inspecting: data/external/reranking-matrices/Reranking/large/2024_09_23_19_33_41_flan-t5-large_webis-touche2020.pkl


,query_id,docs_in_matrix,docs_in_qrels,docs_only_in_qrels,ordered_entries,missing_ordered_union,single_direction_pairs
38,3,100,47,40,9900,9560,0
9,12,100,50,39,9900,9282,0
1,7,100,49,39,9900,9282,0
43,48,100,49,39,9900,9282,0
5,44,100,52,38,9900,9006,0
4,8,100,44,38,9900,9006,0
41,40,100,44,38,9900,9006,0
10,14,100,48,37,9900,8732,0
12,5,100,47,36,9900,8460,0
30,24,100,46,36,9900,8460,0
